# Interactive Optical Bench
## Stops, entrance/exit pupils, focal lengths, and Cassegrain telescopes

**Public browser edition — 8 live optical laboratories.** Move the sliders and the Python ray diagrams and calculated values update automatically.

* Run the import cell first, then run each laboratory cell with **Shift+Enter**.
* The models are **paraxial/first-order** (small angles, ideal lens or mirror); they are educational, not a replacement for Zemax/Code V.
* Signs are explained locally in each example. A **physical aperture stop** is a real opening; an **entrance or exit pupil** is its *optical image* seen from the corresponding side.
* **Effective focal length (EFL)** determines image scale. Back focal length is measured from the final optical surface or a designated reference plane and is generally different.
* Each visitor runs an independent, temporary JupyterLab session through Binder. Save a local copy if you want to keep changes.

### Optical definitions
- **Aperture stop (AS):** physical element setting the on-axis light-cone size.
- **Field stop (FS):** limits which field angles or object points are admitted.
- **Entrance pupil (EP):** the image of AS formed by all optics *before* it, viewed from object space.
- **Exit pupil (XP):** the image of AS formed by all optics *after* it, viewed from image space.
- **Effective focal length:** for an imaging system in air, the first-order relation is \(\lvert y_\mathrm{image}\rvert\simeq \lvert f_\mathrm{eff}\rvert \tan\theta\) for a distant source.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

def interactive(func, **sliders):
    control = widgets.interactive(func, **sliders)
    control.layout = widgets.Layout(width="100%")
    display(control)

def slider(value, min, max, step, title):
    return widgets.FloatSlider(
        value=value, min=min, max=max, step=step, description=title,
        continuous_update=False,
        style={'description_width':'145px'},
        layout=widgets.Layout(width='480px')
    )

def setup(ax, title, xlabel="Optical axis x (mm)"):
    ax.axhline(0, color="0.30", linewidth=0.8)
    ax.grid(True, alpha=0.2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Ray height y (mm)")
    ax.set_title(title)

def imgdist(f, s):
    if abs(s-f) < 1e-8:
        return np.inf
    return f*s/(s-f)

def draw_lens(ax, x, diameter, label="L", color="#1667ad"):
    ax.plot([x,x],[-diameter/2,diameter/2],color=color,lw=3)
    ax.annotate(label,xy=(x,diameter/2),xytext=(x+4,diameter/2+3),color=color)

def draw_stop(ax, x, diam, label="AS"):
    H=45
    ax.plot([x,x],[diam/2,H],lw=4,color="black")
    ax.plot([x,x],[-H,-diam/2],lw=4,color="black")
    ax.annotate(label,xy=(x,H),xytext=(x+4,H-5))
print("Ready: run a laboratory cell to display sliders and diagrams.")

---
## Lab 1 — Thin lens: focal length versus real image position

For a positive thin lens with object distance \(s\) and focal length \(f\),
\[
\frac{1}{s}+\frac{1}{s'}=\frac{1}{f},\qquad
m=-\frac{s'}{s},\qquad D_\mathrm{aperture}=\frac{|f|}{N}
\]
where \(s'\) is the image distance, \(m\) is the lateral magnification and \(N\) the f-number. The **effective focal length remains \(f\)** when you move the object, even though the image distance changes. The horizontal ray leaving the object top, and the ray through the lens center, meet at the paraxial image.

In [ ]:
def lab1(f=80.,s=180.,h=16.):
    sp=imgdist(f,s)
    fig,ax=plt.subplots(figsize=(10,4.3))
    setup(ax,"Thin positive lens: ray construction")
    draw_lens(ax,0,75)
    ax.axvline(f, ls="--",color="tomato",alpha=.7,label="Back focal plane")
    ax.scatter([-s],[h],s=45,color="darkgreen",label="Object top",zorder=5)
    ax.plot([-s,0],[h,h],color="#1d66b0")
    ax.plot([-s,0],[h,0],color="#d18a00")
    xmax=350
    if np.isfinite(sp):
        xp=np.clip(sp,-100,xmax)
        ax.plot([0,xp],[h,h*(1-xp/f)],color="#1d66b0")
        ax.plot([0,xp],[0,-xp*h/s],color="#d18a00")
        if -100<sp<xmax:
            ax.scatter([sp],[-h*sp/s],color="purple",s=45,zorder=6,label="Image top")
        info=f"Object s = {s:.1f} mm | image s' = {sp:.1f} mm | magnification = {-sp/s:.3f}"
    else:
        xx=np.linspace(0,xmax,20)
        ax.plot(xx,h*(1-xx/f),color="#1d66b0")
        ax.plot(xx,-h*xx/s,color="#d18a00")
        info="Object at front focal plane: outgoing rays are parallel (image at infinity)."
    ax.set_xlim(-390,360);ax.set_ylim(-75,75);ax.legend(loc="upper right")
    plt.show()
    print(f"EFL = {f:.1f} mm. {info}")
interactive(lab1,f=slider(80,35,160,5,"Lens f (mm)"),
            s=slider(180,45,350,5,"Object distance (mm)"),
            h=slider(16,5,30,1,"Object height (mm)"))

---
## Lab 2 — Thick lens: effective focal length, principal planes and BFL

A real lens has two refracting surfaces separated by a finite thickness. With the reduced ray vector \((y,n\theta)\), a surface has power \(\Phi=(n_2-n_1)/R\). Combine **refraction → translation → refraction**. In air, if the matrix is \(\begin{bmatrix}A&B\\C&D\end{bmatrix}\), then \(f_\mathrm{EFL}=-1/C\) and back focal length from the **rear vertex** is \(\mathrm{BFL}=-A/C\). They are not generally equal.

In [ ]:
def lab2(n=1.5,t=20.,R1=60.,R2mag=60.):
    def refract(n1,n2,R):
        return np.array([[1.,0.],[-(n2-n1)/R,1.]])
    def travel(d,n):
        return np.array([[1.,d/n],[0.,1.]])
    M=refract(n,1.,-R2mag)@travel(t,n)@refract(1.,n,R1)
    A,B,C,D=M.ravel()
    f=-1/C if abs(C)>1e-9 else np.inf
    bfl=-A/C if abs(C)>1e-9 else np.inf
    h1=(D-1)/C if abs(C)>1e-9 else np.inf
    h2=(1-A)/C if abs(C)>1e-9 else np.inf
    fig,ax=plt.subplots(figsize=(10,3.9))
    setup(ax,"Thick lens: principal-plane and back-focus positions")
    ax.axvspan(0,t,color="skyblue",alpha=.35,label="Glass")
    for x in (0,t): ax.axvline(x,color="steelblue",lw=2)
    if np.isfinite(f):
        ax.axvline(h1,color="green",ls="--",label="H1")
        ax.axvline(t+h2,color="darkorange",ls="--",label="H2")
        if abs(bfl)<600: ax.axvline(t+bfl,color="purple",ls=":",label="Back focus")
        for y0 in (12,22,32):
            ax.plot([-100,0],[y0,y0],color="#1d66b0")
            yv=A*y0; slope=C*y0
            ax.plot([t,t+max(-100,min(350,bfl+30))],[yv,yv+slope*max(-100,min(350,bfl+30))],color="#1d66b0")
    ax.set_ylim(-65,65);ax.set_xlim(min(-110,h1-15), max(250,t+min(350,abs(bfl)+45)))
    ax.legend(loc="upper right",ncol=4);plt.show()
    print(f"EFL = {f:.2f} mm | BFL = {bfl:.2f} mm | H1 at x = {h1:.2f} mm | H2 at x = {t+h2:.2f} mm")
interactive(lab2,n=slider(1.5,1.30,1.85,.05,"Refractive index"),
            t=slider(20,2,45,1,"Thickness (mm)"),
            R1=slider(60,35,150,5,"Front radius (mm)"),
            R2mag=slider(60,35,150,5,"|Rear radius| (mm)"))

---
## Lab 3 — Concave mirror: curvature versus focal length

In the paraxial approximation, a concave spherical mirror has \(f=R/2\). Unlike a lens, light returns toward the side it came from. **Physical mirror diameter changes the f-number** \(f/D\) but not the ideal paraxial focal length.

In [ ]:
def lab3(R=240.,D=70.):
    f=R/2
    fig,ax=plt.subplots(figsize=(10,4))
    setup(ax,"Concave mirror: parallel rays reflect to the prime focus")
    ax.plot([0,0],[-D/2,D/2],color="gray",lw=6,label="Primary mirror at x=0")
    for y in np.linspace(-D*.40,D*.40,7):
        ax.plot([-1.6*f,0],[y,y],color="#1d66b0",alpha=.9)
        ax.plot([0,-f],[y,0],color="#d18a00",alpha=.9)
    ax.scatter([-f],[0],color="red",s=50,zorder=5,label=f"Focus at x = {-f:.1f}")
    ax.set_xlim(-1.7*f,18);ax.set_ylim(-max(D,40)*.66,max(D,40)*.66)
    ax.legend(loc="upper left");plt.show()
    print(f"Radius R = {R:.1f} mm | mirror f = {f:.1f} mm | f-number = f/D = {f/D:.2f}")
interactive(lab3,R=slider(240,80,500,10,"Curvature R (mm)"),
            D=slider(70,25,150,5,"Mirror diameter (mm)"))

---
## Lab 4 — Aperture stop versus field stop

For the illustrated arrangement, an **aperture near the lens** limits the on-axis beam diameter, while an **opening at the focal surface** determines the maximum admitted field angle. For a source at infinity, focal-plane image height is approximately \(y=f\tan\alpha\). The actual stop classification in a complicated instrument must be determined by tracing marginal and chief rays; a small-looking opening is not automatically the aperture stop.

In [ ]:
def lab4(Dap=28.,Dfield=12.,angle=2.0):
    f=120.; alpha=np.deg2rad(angle)
    fig,ax=plt.subplots(figsize=(10,4.2))
    setup(ax,"Near-lens aperture versus focal-plane field stop")
    draw_lens(ax,0,80)
    draw_stop(ax,8,Dap,"Aperture")
    draw_stop(ax,f,Dfield,"Field")
    xs=np.array([-90,0,8,f])
    for h in np.linspace(-34,34,11):
        ypre=h+xs[0]*np.tan(alpha)
        y0=h
        theta=np.tan(alpha)-h/f
        y8=y0+8*theta; yf=y0+f*theta
        passes_ap=abs(y8)<=Dap/2; passes_fs=abs(yf)<=Dfield/2
        col="seagreen" if (passes_ap and passes_fs) else "indianred"
        ax.plot([-90,0],[ypre,y0],color=col,alpha=.7)
        ax.plot([0,8],[y0,y8],color=col,alpha=.7)
        if passes_ap: ax.plot([8,f],[y8,yf],color=col,alpha=.7)
    ax.axvline(f,ls=":",color="purple",alpha=.5)
    ax.set_xlim(-100,155);ax.set_ylim(-50,50);plt.show()
    field_max=np.rad2deg(np.arctan((Dfield/2)/f))
    print(f"Approx. focal image height = {f*np.tan(alpha):.2f} mm | field-stop half-angle = ±{field_max:.2f}°")
    print("Green rays pass both openings; red rays are blocked by at least one opening.")
interactive(lab4,Dap=slider(28,8,65,2,"Aperture Ø (mm)"),
            Dfield=slider(12,2,40,1,"Field stop Ø (mm)"),
            angle=slider(2,-8,8,.5,"Field angle (deg)"))

---
## Lab 5 — The entrance pupil is an *image* of the physical stop

Place a thin positive lens at \(x=0\) and a real stop a distance \(s\) behind it. Viewed from the **left**, the stop is imaged by the lens. Using the thin-lens equation in the reverse direction,
\[
s'=\frac{fs}{s-f},\qquad x_\mathrm{EP}=-s',\qquad D_\mathrm{EP}=D_\mathrm{stop}\left|\frac{s'}{s}\right|.
\]
If \(s>f\), the pupil is a **real image to the left**; if \(s<f\), the pupil is a **virtual image to the right**. At \(s=f\), the entrance pupil is at infinity (object-space telecentric condition). This is not the focal point of the entire system.

In [ ]:
def lab5(f=100.,s=150.,D=20.):
    sp=imgdist(f,s)
    fig,ax=plt.subplots(figsize=(10,4.0))
    setup(ax,"Image of a rear aperture stop seen through a front lens")
    draw_lens(ax,0,72)
    draw_stop(ax,s,D,"Real aperture stop")
    if np.isfinite(sp):
        ep=-sp; dep=D*abs(sp/s)
        if abs(ep)<850:
            ax.plot([ep,ep],[-min(dep,80)/2,min(dep,80)/2],
                    lw=5,ls="--",color="purple",label="Entrance pupil image")
            ax.legend(loc="upper left")
        print(f"Stop position x = +{s:.2f} mm | Entrance pupil x = {ep:.2f} mm | EP diameter = {dep:.2f} mm")
        print("Entrance pupil is", "REAL (object space)" if ep<0 else "VIRTUAL (behind front lens)")
    else:
        print("Entrance pupil is AT INFINITY: chief rays are object-space telecentric.")
    ax.set_xlim(-550,max(s+70,350));ax.set_ylim(-48,48)
    plt.show()
interactive(lab5,f=slider(100,40,180,10,"Front lens f (mm)"),
            s=slider(150,30,260,5,"Stop at x (mm)"),
            D=slider(20,5,45,1,"Stop Ø (mm)"))

---
## Lab 6 — Two-lens EFL versus pupil locations

Two thin lenses of focal lengths \(f_1,f_2\) separated by \(d\) have first-order system power
\[
\frac{1}{f_\mathrm{EFL}}=\frac{1}{f_1}+\frac{1}{f_2}-\frac{d}{f_1f_2}.
\]
A stop between the lenses has **two different images**:
- the **entrance pupil**, made by the first lens when looking into the instrument from the left;
- the **exit pupil**, made by the second lens when looking from the right.

Move the stop and compare: the **EFL does not change**, but the **pupil positions and sizes do**. Adjust \(d\simeq f_1+f_2\) to approach an **afocal** system.

In [ ]:
def lab6(f1=120.,f2=100.,d=80.,fraction=.5,D=12.):
    a=d*fraction
    M=np.array([[1.,0.],[-1/f2,1.]])@np.array([[1.,d],[0.,1.]])@np.array([[1.,0.],[-1/f1,1.]])
    A,B,C,DD=M.ravel()
    efl=-1/C if abs(C)>1e-10 else np.inf
    bfl=-A/C if abs(C)>1e-10 else np.inf
    s1=imgdist(f1,a); s2=imgdist(f2,d-a)
    ep=-s1 if np.isfinite(s1) else np.inf
    xp=d+s2 if np.isfinite(s2) else np.inf
    dep=D*abs(s1/a) if np.isfinite(s1) else np.inf
    dxp=D*abs(s2/(d-a)) if np.isfinite(s2) else np.inf
    fig,ax=plt.subplots(figsize=(11,4))
    setup(ax,"A physical stop and its two optical pupil images")
    draw_lens(ax,0,65,"L1");draw_lens(ax,d,65,"L2")
    draw_stop(ax,a,D,"Physical AS")
    for x,diam,color,name in [(ep,dep,"purple","EP"),(xp,dxp,"orange","XP")]:
        if np.isfinite(x) and abs(x)<900:
            ax.plot([x,x],[-min(diam,76)/2,min(diam,76)/2],
                    ls="--",lw=4,color=color,label=name)
    ax.set_xlim(min(-220,ep-30 if np.isfinite(ep) and ep>-650 else -220),
                max(d+240,xp+30 if np.isfinite(xp) and xp<650 else d+240))
    ax.set_ylim(-50,50);ax.legend(loc="upper right");plt.show()
    def fmt(x): return f"{x:.2f}" if np.isfinite(x) else "infinity / afocal"
    print("System EFL =",fmt(efl),"mm | BFL from lens 2 =",fmt(bfl),"mm")
    print("Entrance pupil x =",fmt(ep),"mm; Ø =",fmt(dep),"mm")
    print("Exit pupil x =",fmt(xp),"mm; Ø =",fmt(dxp),"mm")
interactive(lab6,f1=slider(120,60,210,10,"Lens 1 f (mm)"),
            f2=slider(100,50,200,10,"Lens 2 f (mm)"),
            d=slider(80,40,240,10,"Separation (mm)"),
            fraction=slider(.5,.15,.85,.05,"Stop position / d"),
            D=slider(12,4,35,1,"Stop Ø (mm)"))

---
## Lab 7 — Afocal Kepler telescope and exit pupil

A Keplerian telescope has objective focal length \(f_o\), eyepiece focal length \(f_e\), and spacing \(d=f_o+f_e\). Its **angular magnification** is \(-f_o/f_e\); its **system EFL is infinite**, because the telescope is afocal. If the objective is the aperture stop, the exit pupil is the image of the objective through the eyepiece, of diameter \(D_\mathrm{exit}=D_o(f_e/f_o)\). Even though EFL is infinite, the *individual* lenses retain their finite focal lengths.

In [ ]:
def lab7(fo=120.,fe=30.,D=42.,alpha=0.):
    d=fo+fe
    sp=imgdist(fe,d)
    xexit=d+sp
    dexit=D*abs(sp/d)
    a=np.deg2rad(alpha)
    fig,ax=plt.subplots(figsize=(10,4.5))
    setup(ax,"Afocal telescope: parallel rays in, parallel rays out")
    draw_lens(ax,0,D,"Objective")
    draw_lens(ax,d,60,"Eyepiece")
    for y0 in np.linspace(-D*.45,D*.45,7):
        theta_in=np.tan(a)
        yeye=y0+d*(theta_in-y0/fo)
        theta_out=(theta_in-y0/fo)-yeye/fe
        ax.plot([-70,0],[y0-70*theta_in,y0],color="#1d66b0",alpha=.8)
        ax.plot([0,d],[y0,yeye],color="#d18a00",alpha=.8)
        ax.plot([d,d+90],[yeye,yeye+90*theta_out],color="seagreen",alpha=.8)
    if xexit<d+160:
        ax.plot([xexit,xexit],[-dexit/2,dexit/2],color="purple",lw=4,ls="--",label="Exit pupil")
    ax.set_xlim(-90,d+180);ax.set_ylim(-75,75);ax.legend(loc="upper right")
    plt.show()
    print(f"Angular magnification = {-fo/fe:.2f}× | Objective-to-eyepiece spacing = {d:.1f} mm")
    print(f"Exit pupil: x = {xexit:.2f} mm, Ø = {dexit:.2f} mm")
    print("EFL of the complete afocal telescope = infinity.")
interactive(lab7,fo=slider(120,80,240,10,"Objective f (mm)"),
            fe=slider(30,20,90,5,"Eyepiece f (mm)"),
            D=slider(42,20,75,5,"Objective Ø (mm)"),
            alpha=slider(0,-1,1,.1,"Field angle (deg)"))

---
## Lab 8 — Classical Cassegrain: EFL is not the back focal distance

Set the primary mirror at \(x=0\), the convex secondary at \(x=-d\), and the final focus behind the primary at \(x=+b\). The **primary focal length** \(f_1>0\) describes where the primary alone would focus incoming collimated light (\(x=-f_1\)).

Before reaching the prime focus, the converging beam hits the convex secondary; the secondary magnification is
\[
m_2=\frac{d+b}{f_1-d},\qquad f_\mathrm{EFL}=m_2 f_1,\qquad F\#=f_\mathrm{EFL}/D_1.
\]
For a small distant object field angle \(\alpha\), the final image height is approximately \(f_\mathrm{EFL}\tan\alpha\).

**Example:** \(f_1=800\) mm, \(d=600\) mm, \(b=150\) mm gives \(m_2=3.75\), \(f_\mathrm{EFL}=3000\) mm even though the final focus is only 150 mm *behind the primary*. A Ritchey–Chrétien alters both mirror conic constants to correct third-order spherical aberration and coma; these paraxial geometry equations still apply.

The drawing uses the *unfolded paraxial rays between reflection points in real positions*, not full mirror surface sag or off-axis aberrations.

In [ ]:
def lab8(f1=800.,dfrac=.75,b=150.,D1=200.,alpha_arcmin=1.):
    d=f1*dfrac
    delta=f1-d
    m2=(d+b)/delta
    F=m2*f1
    image_height=F*np.tan(np.deg2rad(alpha_arcmin/60.))
    f2=1/(-1/delta+1/(d+b)) if abs(-1/delta+1/(d+b))>1e-12 else np.inf
    fig,ax=plt.subplots(figsize=(11,5))
    setup(ax,"Cassegrain: rays reflect at the primary and secondary")
    ax.plot([0,0],[-D1/2,D1/2],color="gray",lw=7,label="Primary M1")
    D2=D1*(1-d/f1)*1.4
    ax.plot([-d,-d],[-D2/2,D2/2],color="purple",lw=6,label="Convex secondary M2")
    for h in np.linspace(-D1*.4,D1*.4,9):
        hs=h*(1-d/f1)
        ax.plot([-d-120,0],[h,h],color="#1667ad",alpha=.65)
        ax.plot([0,-d],[h,hs],color="#e0a000",alpha=.75)
        ax.plot([-d,b],[hs,0],color="seagreen",alpha=.75)
    ax.plot([-f1,-f1],[-D1*.18,D1*.18],ls="--",color="tomato",lw=1.5,label="M1 prime focus")
    ax.scatter([b],[0],s=60,color="black",label="Final focus")
    ax.set_xlim(-max(f1+90,d+170),max(b+120,210))
    ax.set_ylim(-D1*.62,D1*.62);ax.legend(loc="upper right",ncol=2)
    plt.show()
    print(f"Secondary magnification = {m2:.3f}×; primary f = {f1:.1f} mm; secondary paraxial f = {f2:.2f} mm")
    print(f"System EFL = {F:.2f} mm; f-number = {F/D1:.2f}; mechanical back focus = {b:.2f} mm")
    print(f"At {alpha_arcmin:.2f} arcmin, first-order image height ≈ {image_height:.3f} mm.")
    print("Entrance pupil ≈ primary mirror when the primary is the true aperture stop.")
interactive(lab8,f1=slider(800,400,1400,50,"M1 focal f1 (mm)"),
            dfrac=slider(.75,.50,.92,.02,"Separation / f1"),
            b=slider(150,20,350,10,"Back focus b (mm)"),
            D1=slider(200,100,350,10,"Primary Ø (mm)"),
            alpha_arcmin=slider(1,0,5,.5,"Field (arcmin)"))

---
## Follow-up questions and limitations

1. For the two-lens experiment, move the stop but keep the lenses fixed: does **system EFL** change? What does?
2. In the entrance pupil experiment, what happens as the physical stop passes through the front lens focal plane?
3. In the Cassegrain experiment, hold \(f_1\) and \(D_1\) fixed while increasing the secondary separation: how do magnification and f-number change?
4. Why can a telescope have a 3 m effective focal length but only a 150 mm back focus?
5. Which pupil does an eyepiece need to match to the observer's eye pupil?

**Model boundaries:** No real surface shapes, chromatic dispersion, obscuration diffraction, vignetting from mechanical baffles, exact pupil aberrations, RC conic-constant optimization, or higher-order aberrations are included. Chief rays and stop changes in real optical instruments require ray tracing at the relevant object field and aperture. The Cassegrain plot idealizes the convex secondary as a plane at a representative axial position.

**References:** Hecht, *Optics*; Smith, *Modern Optical Engineering*; Schroeder, *Astronomical Optics*; Wilson, *Reflecting Telescope Optics*.
